In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from torch.optim import Adam
import os
import torch.multiprocessing as mp
import torch.distributed as dist


os.environ["CUDA_VISIBLE_DEVICES"] = "3"


def setup(rank, world_size):
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "11030"    
    dist.init_process_group("nccl", rank=rank, world_size=world_size)

def cleanup():
    dist.destroy_process_group()

# 加载数据集
data_folder = './FMNIST_data'
train_data = datasets.FashionMNIST(data_folder, download=False, train=True, transform=None)
test_data = datasets.FashionMNIST(data_folder, download=False, train=False, transform=None)

imgs = train_data.data
labels = train_data.targets

class FMINISTDataset(Dataset):
    def __init__(self, x, y):
        x = x.float()/255
        x = x.view(-1, 28 * 28)
        self.x, self.y = x, y
        
    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, idx):
        x, y = self.x[idx], self.y[idx]
        return x, y
    
def get_data():
    data = FMINISTDataset(imgs, labels)
    partial_data = DataLoader(data, batch_size=64, shuffle=True)
    return partial_data

class MyNet(nn.Module):
    def __init__(self):
        super(MyNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(28*28, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
    
    def forward(self, x):
        x = self.model(x)
        return x
 
def loss_function():
    return nn.CrossEntropyLoss()

@torch.no_grad()
def accuracy(x, y, model):
    model.eval()
    y_pred = model(x)
    # y_pred是10类对应的概率，取概率最大的类别pred_class
    max_val, pred_class = y_pred.max(-1)
    is_correct = (pred_class == y)
    return is_correct.float().mean().item()

@torch.no_grad()
def val_loss(model, x, y, criterion):
    y_pred = model(x)
    loss = criterion(y_pred, y)
    return loss.item()
    

def train(rank, world_size):
    setup(rank, world_size)
    
    # 1. 准备数据（使用 DistributedSampler）
    dataset = FMINISTDataset(imgs, labels)
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank)
    dataloader = DataLoader(dataset, batch_size=64, sampler=sampler)

    # 2. 初始化模型并包装为 DDP
    model = MyNet().to(rank)
    model = nn.parallel.DistributedDataParallel(model, device_ids=[rank])

    # 3. 定义优化器和损失函数
    criterion = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=0.001)

    # 4. 训练循环
    for epoch in range(10):
        sampler.set_epoch(epoch)  # 确保每个 epoch 数据顺序不同
        model.train()
        epoch_loss, epoch_acc = 0.0, 0.0
        
        for batch_idx, (x, y) in enumerate(dataloader):
            x, y = x.to(rank), y.to(rank)  # 数据移动到当前 GPU
            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

            # 计算准确率
            _, preds = torch.max(outputs, 1)
            correct = (preds == y).float().mean().item()
            
            epoch_loss += loss.item()
            epoch_acc += correct

        # 打印主进程（rank=0）的训练结果
        if rank == 0:
            avg_loss = epoch_loss / len(dataloader)
            avg_acc = epoch_acc / len(dataloader)
            print(f"Epoch [{epoch+1}/10], Loss: {avg_loss:.4f}, Accuracy: {avg_acc:.4f}")

    # 训练结束后保存模型（仅主进程）
    if rank == 0:
        torch.save(model.module.state_dict(), "fashion_mnist_model.pth")
    
    cleanup()

if __name__ == "__main__":
    world_size = 3  # 使用GPU数量
    mp.spawn(train, args=(world_size,), nprocs=world_size, join=True)
